## DS6015-006 
## Derrick Clarke (thq3hn)

### Generating a Canopy Height Model (CHM) from LiDAR Point Cloud Data

#### A Canopy Height Model is a raster (gridded image) where each pixel value represents the height of the vegetation above the ground at that location. It is computed as the difference between two other raster surfaces:

    - CHM = Digital Surface Model (DSM) − Digital Terrain Model (DTM)

#### The DSM captures the elevation of the highest return at each location (i.e., the top of the canopy or any other surface feature). The DTM captures the elevation of the bare ground. Subtracting the ground from the surface isolates vegetation height.

### Install Required Libraries

In [1]:
#%pip install laspy lazrs-python numpy scipy rasterio matplotlib pyproj
#Did not work for me.

#### This Worked 
(DS6001_PY311_ENV) ddclarke@GSLAL0325070029 ~ % pip install laspy lazrs-python                                       
Collecting laspy
  Using cached laspy-2.7.0-py3-none-any.whl.metadata (3.4 kB)
ERROR: Could not find a version that satisfies the requirement lazrs-python (from versions: none)
ERROR: No matching distribution found for lazrs-python
(DS6001_PY311_ENV) ddclarke@GSLAL0325070029 ~ % python --version                                                     
Python 3.11.6
(DS6001_PY311_ENV) ddclarke@GSLAL0325070029 ~ % pip install "laspy[lazrs,laszip]"
Collecting laspy[laszip,lazrs]
  Using cached laspy-2.7.0-py3-none-any.whl.metadata (3.4 kB)
Requirement already satisfied: numpy in /opt/homebrew/Caskroom/miniconda/base/envs/DS6001_PY311_ENV/lib/python3.11/site-packages (from laspy[laszip,lazrs]) (2.1.3)
Collecting laszip<0.4.0,>=0.3.0 (from laspy[laszip,lazrs])
  Downloading laszip-0.3.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.8 kB)
Collecting lazrs<0.9.0,>=0.8.0 (from laspy[laszip,lazrs])
  Downloading lazrs-0.8.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (75 bytes)
Downloading laspy-2.7.0-py3-none-any.whl (86 kB)
Downloading laszip-0.3.0-cp311-cp311-macosx_11_0_arm64.whl (264 kB)
Downloading lazrs-0.8.1-cp311-cp311-macosx_11_0_arm64.whl (590 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.7/590.7 kB 3.3 MB/s eta 0:00:00
Installing collected packages: lazrs, laszip, laspy
Successfully installed laspy-2.7.0 laszip-0.3.0 lazrs-0.8.1
(DS6001_PY311_ENV) ddclarke@GSLAL0325070029 ~ % 

In [2]:
import laspy
import numpy as np

from scipy.interpolate import griddata

import rasterio
from rasterio.transform import from_origin


### Load the LiDAR Point Cloud

In [3]:

# Load a single VGIN LiDAR tile
#las = laspy.read("../../data/lidar_tiles_central_va/USGS_LPC_VA_Central_Virginia_Seismic_LiDAR_2013_14_17SQB485965.laz")
#las = laspy.read("../../data/laz_files/USGS_LPC_VA_ChesapeakeBay_2015_LAS_S13_4990_20.laz")
las = laspy.read("../../data/laz_files/USGS_LPC_VA_ChesapeakeBay_2015_LAS_S13_4889_10.laz")

# Extract X, Y, Z coordinates
x: np.ndarray = las.x.scaled_array()   # Easting (meters, projected CRS)
y: np.ndarray = las.y.scaled_array()   # Northing (meters, projected CRS)
z: np.ndarray = las.z.scaled_array()   # Elevation (meters above sea level)

# Extract classification codes
# ASPRS standard: 2 = Ground, 5 = High Vegetation
classification: np.ndarray = np.array(las.classification)

print(f"Total points: {len(x):,}")
print(f"Ground points: {np.sum(classification == 2):,}")
print(f"Vegetation points: {np.sum(classification == 5):,}")


Total points: 18,773,900
Ground points: 7,014,867
Vegetation points: 0


## How can this be???  Zero (0) Vegetation Points

There are zero points classified as 3 (Low Vegetation), 4 (Medium Vegetation), or 5 (High Vegetation) anywhere in this tile. It's not that your filtering logic is missing them — the classes simply don't exist in this file. Over 55% of all points are dumped into class 1 (Unclassified), which is almost certainly where your vegetation returns actually are, just not labeled as such.
Why: this is a vintage/spec issue, not a bug
This file is from a 2015 USGS 3DEP LPC delivery. Current USGS lidar requirements mandate a full classification scheme — the minimum required classification scheme for lidar data is defined in the current specification's table, and all points falling within that scheme must be properly classified, with no points left in class 0 unless withheld. Critically, classes beyond the minimum required scheme are optional per-project additions, and their accuracy isn't assessed by USGS unless a partner specifically requested them. USGS

Older Lidar Base Specification versions (the ones in force around 2015) had a much smaller minimum scheme — often just Ground (2) + Unclassified (1), with vegetation/building/water stratification treated as an optional add-on that not every contract paid for. What you're looking at is consistent with a delivery where only bare-earth ground classification was contractually required; everything else (trees, buildings, understory, etc.) was left in class 1.


### VGIN LiDAR tiles follow the ASPRS LAS classification standard. The key classes for CHM generation are:

Class Code  Meaning
1           Unclassified
2           Ground
3           Low vegetation (< 0.5m)
4           Medium vegetation (0.5 - 2m)
5           High vegetation (> 2m)
6           Building

### Separate Ground and Vegetation Points. (1st Pass)

In [4]:
# Ground points only (for DTM)
ground_mask = classification == 2
x_ground = x[ground_mask]
y_ground = y[ground_mask]
z_ground = z[ground_mask]

# All vegetation points (for DSM — use all returns above ground)
veg_mask = classification >= 3
x_veg = x[veg_mask]
y_veg = y[veg_mask]
z_veg = z[veg_mask]


### Separate Ground and Vegetation Points. (2nd Pass)

We need to derive vegetation from height-above-ground, since the ground surface (class 2) is present and reliable:

In [9]:
second_pass_ground_mask = las.classification == 2

gx, gy, gz = las.x[second_pass_ground_mask], las.y[second_pass_ground_mask], las.z[second_pass_ground_mask]

# Interpolate a ground surface (DTM), then compute height above it
# for all non-ground points
second_pass_nonground_mask = las.classification == 1
nx, ny, nz = las.x[second_pass_nonground_mask], las.y[second_pass_nonground_mask], las.z[second_pass_nonground_mask]

second_pass_ground_z_at_points = griddata((gx, gy), gz, (nx, ny), method="linear")
second_pass_height_above_ground = nz - second_pass_ground_z_at_points

# Standard ASPRS-style height thresholds (tune to your use case)
low_veg    = (second_pass_height_above_ground > 0.15) & (second_pass_height_above_ground <= 2.0)
med_veg    = (second_pass_height_above_ground > 2.0)  & (second_pass_height_above_ground <= 5.0)
high_veg   = (second_pass_height_above_ground > 5.0)

### Define the Raster Grid

In [5]:
# Define output resolution in meters (1 m is standard for VGIN data)
resolution = 1.0

# Compute grid extent
x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()

# Number of columns and rows
ncols = int(np.ceil((x_max - x_min) / resolution))
nrows = int(np.ceil((y_max - y_min) / resolution))

print(f"Grid size: {nrows} rows x {ncols} cols")


Grid size: 5000 rows x 5000 cols


### Rasterize Ground Points into a DTM

In [6]:
# Create a regular grid of coordinates
grid_x = np.linspace(x_min, x_max, ncols)
grid_y = np.linspace(y_min, y_max, nrows)
grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)

# Interpolate ground elevations onto the grid
# Use 'linear' for speed; 'cubic' for smoother results
dtm = griddata(
    points=np.column_stack([x_ground, y_ground]),
    values=z_ground,
    xi=(grid_xx, grid_yy),
    method='linear',
    fill_value=np.nan
)

print(f"DTM shape: {dtm.shape}, min: {dtm.min():.2f} m, max: {dtm.max():.2f} m")

DTM shape: (5000, 5000), min: nan m, max: nan m


### Rasterize Vegetation Points into a DSM (Maximum Return)

In [7]:
# Initialize DSM with -infinity (we want the MAXIMUM Z per cell)
dsm = np.full((nrows, ncols), -np.inf)

# Convert point coordinates to grid cell indices
col_idx = ((x_veg - x_min) / resolution).astype(int)
row_idx = ((y_max - y_veg) / resolution).astype(int)  # Note: Y axis is flipped

# Clip indices to grid bounds
valid = (col_idx >= 0) & (col_idx < ncols) & (row_idx >= 0) & (row_idx < nrows)
col_idx = col_idx[valid]
row_idx = row_idx[valid]
z_veg_valid = z_veg[valid]

# Assign maximum Z value to each cell
np.maximum.at(dsm, (row_idx, col_idx), z_veg_valid)

# Replace -inf (no vegetation return) with NaN
dsm[dsm == -np.inf] = np.nan


### Compute the CHM

In [8]:
# CHM = DSM - DTM
chm = dsm - dtm

# Clip negative values (artefacts from interpolation errors)
chm = np.where(chm < 0, 0, chm)

# Optionally clip unrealistic heights (e.g., > 60 m for Central Virginia)
chm = np.where(chm > 60, np.nan, chm)

print(f"CHM shape: {chm.shape}")
print(f"Max canopy height: {np.nanmax(chm):.1f} m")
print(f"Mean canopy height: {np.nanmean(chm):.1f} m")


CHM shape: (5000, 5000)
Max canopy height: 60.0 m
Mean canopy height: 10.4 m


### Save the CHM as a GeoTIFF

In [9]:
# Define the affine transform (maps pixel indices to real-world coordinates)
transform = from_origin(x_min, y_max, resolution, resolution)

# Write to GeoTIFF
with rasterio.open(
    "../../data/outputs/GeoTIFF_files/chm_output.tif",
    mode="w",
    driver="GTiff",
    height=nrows,
    width=ncols,
    count=1,
    dtype=chm.dtype,
    crs="EPSG:32618",   # UTM Zone 18N — standard for Central Virginia
    transform=transform,
    nodata=np.nan
) as dst:
    dst.write(chm, 1)

print("CHM saved to ../../data/outputs/GeoTIFF_files/chm_output.tif")

CHM saved to ../../data/outputs/GeoTIFF_files/chm_output.tif
